# Add the MPC designation to the Rubin observations
#### *Open Tools, Open Sky: Doing LSST Science with Public Rubin Data & LINCC Frameworks, UdeA, Medellín, October 13 2026*

Each Rubin detection submitted to the Minor Planet Center is identified by its `permid` (number) and/or `provid`
(provisional designation). Here we find, for every detection, the readable designation of the object in the MPC
orbit catalogue (MPCORB `designation`, e.g. `(1) Ceres`, `(800001) 2013 YW88` or `2021 AQ`) and save it as a new
column, `mpc_designation`. Afterwards the orbits can be added to the observations with a simple join on the designation
(see `exploration_data.ipynb`).

In [1]:
import pandas as pd
import numpy as np

# Vera C. Rubin Observatory observations submitted to the Minor Planet Center
data = pd.read_parquet('data/obs_sbn_X05_full.parquet', engine='pyarrow')
print(f'{len(data):,} detections')

8,078,546 detections


## MPCORB designations

We only need the designation fields of `data/mpcorb_extended.dat.gz` (same fixed-width format as in `exploration_data.ipynb`):
the readable `designation` and, from the extended version, the `other_desigs` of each object.

In [2]:
import gzip

path = 'data/mpcorb_extended.dat.gz'
colspecs = {
    'designation':  (166, 194),   # readable designation, e.g. '(1) Ceres' or '2021 AQ'
    'other_desigs': (217, 400),   # extended: other designations of the same object
}

# The data start after the header's line of dashes
with gzip.open(path, 'rt') as f:
    for i, line in enumerate(f):
        if line.startswith('-----'):
            header_lines = i + 1
            break

mpcorb = pd.read_fwf(path, compression='gzip', skiprows=header_lines, header=None,
                     colspecs=list(colspecs.values()), names=list(colspecs.keys()),
                     skip_blank_lines=True, dtype_backend='pyarrow')
print(f'{len(mpcorb):,} objects, designations unique: {mpcorb["designation"].is_unique}')

1,573,538 objects, designations unique: True


In [3]:
# Identifiers in the same form as the observation file, so the two can be matched:
# '(1) Ceres' -> permid '1'; '(800001) 2013 YW88' -> permid '800001', provid '2013 YW88'; '2021 AQ' -> provid '2021 AQ'
numbered = mpcorb['designation'].str.extract(r'^\((?P<permid>\d+)\)\s*(?P<rest>.*)$')
rest_is_provid = numbered['rest'].str.match(r'^\d{4} [A-Z]{2}\d*$') == True
mpcorb['permid'] = numbered['permid']
mpcorb['provid'] = mpcorb['designation'].where(mpcorb['permid'].isna(), numbered['rest'].where(rest_is_provid))

## Join the two catalogues

An object in MPCORB can be found by three keys: its number (`permid`), its main provisional designation (`provid`),
or one of its other designations (`other_desigs`). We make one small table `key -> designation` for each key and
join it to the observations with `merge`. The joins are done once per distinct (`permid`, `provid`) pair, not for all
8M detections. Priority: `permid`, then `provid`, then the other designations; `match_by` records which one matched.
Detections with both `permid` and `provid` null (and comets, which are not in MPCORB) get no designation.

In [ ]:
# 1. One table per key: key -> MPC designation
by_permid = mpcorb[['permid', 'designation']].dropna()
by_provid = mpcorb[['provid', 'designation']].dropna()

by_other = mpcorb[['designation', 'other_desigs']].dropna()
by_other['provid'] = by_other['other_desigs'].str.strip().str.split(r'\s{2,}', regex=True)
by_other = by_other.explode('provid')[['provid', 'designation']].dropna()

# A designation should point to one object: drop the (rare) ambiguous ones
by_provid = by_provid.drop_duplicates('provid', keep=False)
by_other = by_other.drop_duplicates('provid', keep=False)

# 2. Join each table to the distinct (permid, provid) pairs of the observations
pairs = data[['permid', 'provid']].drop_duplicates()
pairs = pairs.merge(by_permid.rename(columns={'designation': 'desig_permid'}), on='permid', how='left')
pairs = pairs.merge(by_provid.rename(columns={'designation': 'desig_provid'}), on='provid', how='left')
pairs = pairs.merge(by_other.rename(columns={'designation': 'desig_other'}), on='provid', how='left')

# 3. Keep the first match in order of priority
pairs['mpc_designation'] = pairs['desig_permid'].fillna(pairs['desig_provid']).fillna(pairs['desig_other'])
pairs['match_by'] = np.select([pairs['desig_permid'].notna(), pairs['desig_provid'].notna(), pairs['desig_other'].notna()],
                              ['permid', 'provid', 'other_desig'], default=None)

# 4. Bring the result back to every detection (a left merge keeps the row order;
#    pandas also matches null keys to null keys, so rows with a null permid or provid are handled)
matched = data[['permid', 'provid']].merge(pairs[['permid', 'provid', 'mpc_designation', 'match_by']],
                                           on=['permid', 'provid'], how='left')
data['mpc_designation'] = matched['mpc_designation'].array
data['match_by'] = matched['match_by'].to_numpy()

has_desig = data['mpc_designation'].notna()
print(f"Detections with an MPC designation: {has_desig.sum():,} of {len(data):,} ({100 * has_desig.mean():.2f}%)")
data['match_by'].value_counts(dropna=False)

In [5]:
# Example: 2025 MD38 is a secondary designation of 2014 HR48
data.loc[data['provid'] == '2025 MD38', ['provid', 'permid', 'mpc_designation', 'match_by']].head()

,provid,permid,mpc_designation,match_by
18278,2025 MD38,None,2014 HR48,other_desig
18279,2025 MD38,None,2014 HR48,other_desig
18280,2025 MD38,None,2014 HR48,other_desig
18281,2025 MD38,None,2014 HR48,other_desig
18282,2025 MD38,None,2014 HR48,other_desig


## Save the Rubin catalogue with the new column

In [6]:
out = 'data/obs_sbn_X05_full_mpc_desig.parquet'
data.to_parquet(out, engine='pyarrow', index=False)
print(f'Saved {len(data):,} rows x {data.shape[1]} columns to {out}')

Saved 8,078,546 rows x 92 columns to data/obs_sbn_X05_full_mpc_desig.parquet
